# 01 — Kaggle text-coding pilot (GATED, candidate, engineering diagnostic)

**Status: last real run (2026-09-29) started the server and made 5 real
requests, but produced 0 usable coded trajectories** -- 2 parse/schema
failures, 1 context-limit block. See `docs/experiment_log.md` for the full,
evidence-based account (facts vs suspected causes kept separate) and
`docs/methods_and_deviations.md` for what changed here in response.

**Self-contained by design:** this notebook reads project code and data
ONLY from a private Kaggle Dataset you attach (Section 2) -- never from
GitHub.

**What changed after the 2026-09-29 run, in one paragraph:** every real
request/response is now saved to disk the moment it happens (`urop.evidence`),
independent of whether the trajectory it belongs to ever finishes --
the previous run silently lost an early chunk's real response when a LATER
chunk's repair request got blocked. A new, opt-in **structured JSON output
mode** (`urop.structured_coding`) asks the server to constrain generation to
a JSON Schema instead of a freeform `===CODES===` block, specifically
because the freeform mode's failure was model-generated quotes containing
unescaped nested JSON, which broke `json.loads` outright -- not something a
repair retry can fix. The CUDA-linker workaround needed to start the server
at all (`ld: cannot find -lcuda`) is now a permanent, verified cell instead
of an ad hoc one. Reporting now separates "the loop finished" from "the
loop produced anything usable" (`urop.reporting.build_outcome_report`) --
`stop_reason="completed"` no longer reads as success by itself.

**Scope: this run is a ONE-TRAJECTORY diagnostic**, not a multi-trajectory
smoke test. Section 5's data-loading cell filters the 3-trajectory sample
down to exactly `instance_id="13"`, `run_id="trial_0"` -- the one whose
freeform-mode output was confirmed malformed on 2026-09-29 -- and asserts
exactly one record is selected. This checks structured-output compatibility
and the evidence/reporting pipeline on a KNOWN case, not a fresh sample.
**If this run fails or returns invalid evidence, stop and preserve the
artifacts (Section 9) rather than launching another run.** Any proposed
increase to model size, output tokens, context length, or retries needs to
be justified from recorded evidence, not used to paper over a failure.

**Hard limits, unchanged:**
- One T4 GPU, FP16, concurrency 1. Leave the second T4 unused --
  **but selecting the "GPU T4 x2" accelerator bills your weekly quota for
  BOTH GPUs for the whole session regardless of which one this notebook
  actually touches.** "Unused" describes our own process's behavior, not
  what Kaggle charges you for.
- No paid API calls anywhere in this notebook. `backend` stays `"local"`.
- Every GPU-spending cell has a `try/finally` that terminates the server
  process GROUP on ANY exception -- but you must still run the shutdown
  cell (Section 8) and then Kaggle's own Stop Session control to actually
  end the session. **An ALLOCATED GPU keeps consuming your weekly quota
  for as long as the Kaggle session stays up, whether or not it is doing
  any work at that moment -- "idle" is not "free."**


## 0. Hard gate — do not skip or silently edit past this

In [ ]:
# Flip these to True only after you have actually done each thing.
I_CONFIRMED_KAGGLE_INTERNET_IS_ON = False        # Settings -> Internet -> On
I_CONFIRMED_GPU_ACCELERATOR_IS_T4x2 = False      # Settings -> Accelerator -> GPU T4 x2
I_ATTACHED_THE_PRIVATE_PROJECT_BUNDLE = False    # see Section 2 -- a single private Kaggle Dataset, not GitHub
I_UNDERSTAND_BUDGET_IS_MINUTES_NOT_HOURS = False
I_UNDERSTAND_THIS_IS_A_ONE_TRAJECTORY_DIAGNOSTIC = False  # not a fresh multi-trajectory smoke test -- see intro
I_WILL_RUN_THE_SHUTDOWN_CELL_AT_THE_END = False

_gate = {
    "internet_on": I_CONFIRMED_KAGGLE_INTERNET_IS_ON,
    "gpu_t4x2": I_CONFIRMED_GPU_ACCELERATOR_IS_T4x2,
    "bundle_attached": I_ATTACHED_THE_PRIVATE_PROJECT_BUNDLE,
    "budget_understood": I_UNDERSTAND_BUDGET_IS_MINUTES_NOT_HOURS,
    "one_trajectory_diagnostic_understood": I_UNDERSTAND_THIS_IS_A_ONE_TRAJECTORY_DIAGNOSTIC,
    "will_shutdown": I_WILL_RUN_THE_SHUTDOWN_CELL_AT_THE_END,
}
_failed = [k for k, v in _gate.items() if not v]
if _failed:
    raise RuntimeError(f"Gate not satisfied, refusing to continue: {_failed}.")
print("Gate passed. Every following cell is still a candidate, verify as you go.")


## 1. Environment preflight (no GPU cost yet)

In [ ]:
import json
import os
import platform
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())

total, used, free = shutil.disk_usage("/kaggle/working")
print(f"/kaggle/working disk: {free / 1e9:.1f} GB free of {total / 1e9:.1f} GB")

try:
    smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30)
    print(smi.stdout if smi.returncode == 0 else f"nvidia-smi failed: {smi.stderr}")
except FileNotFoundError:
    print("nvidia-smi not found -- no NVIDIA GPU visible. Stop here if you expected one.")

def check_reachable(url: str, timeout: float = 5.0) -> str:
    try:
        urllib.request.urlopen(url, timeout=timeout)
        return "reachable"
    except Exception as exc:  # noqa: BLE001 -- preflight, report don't raise
        return f"unreachable ({exc})"

for url in ["https://pypi.org", "https://huggingface.co"]:
    print(f"{url}: {check_reachable(url)}")


## 2. Load the project from a private bundle (NOT GitHub) + validate paths

**Fix for a real bug found in review:** the previous version of this
notebook did `git clone` from the public GitHub repo. Local work in this
project is routinely uncommitted/unpushed -- a clone would silently run
whatever happens to be on GitHub, not the code actually being reviewed.
There is no git clone here.

Prepare a **private** Kaggle Dataset containing this project (see
`docs/kaggle_quickstart.md`), attach it under "Add Input", then set
`PROJECT_BUNDLE_INPUT` below to its mount path. This cell copies it into
**`/kaggle/working`** (the only writable location).


In [ ]:
bundle_roots = sorted({
    p.parent.parent.parent for p in Path("/kaggle/input").rglob("__init__.py")
    if p.parent.name == "urop" and p.parent.parent.name == "src"
})
assert len(bundle_roots) == 1, f"Expected exactly one attached private project bundle; found {bundle_roots}."
PROJECT_BUNDLE_INPUT = bundle_roots[0]
REPO_ROOT = Path("/kaggle/working/UROP_Prof_Hong_Atharva")

if not PROJECT_BUNDLE_INPUT.exists():
    raise RuntimeError(
        f"{PROJECT_BUNDLE_INPUT} does not exist. Attach the private project-bundle Kaggle "
        "Dataset (Add Input) and set PROJECT_BUNDLE_INPUT to its real mount path first."
    )

if REPO_ROOT.exists():
    raise RuntimeError(
        "Use a fresh session, or delete REPO_ROOT first -- refusing to silently reuse "
        "potentially STALE source files already copied earlier in this kernel session."
    )

shutil.copytree(PROJECT_BUNDLE_INPUT, REPO_ROOT)
print(f"Copied {PROJECT_BUNDLE_INPUT} -> {REPO_ROOT}")

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "tests"))  # fake_local_server, if you want to smoke-test the wiring first

# All private evidence (raw responses, install logs, per-request JSON) is
# written under /kaggle/working, OUTSIDE REPO_ROOT and outside any git repo
# -- see Section 3/5/9. Never committed, never part of either packaged ZIP.
INSTALL_EVIDENCE_DIR = Path("/kaggle/working/install_evidence")
INSTALL_EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Validate every path Section 4-7 will need BEFORE any GPU/inference work is
# allocated -- fail loudly here, not halfway through a paid GPU session.
REQUIRED_PATHS = {
    "our own package": REPO_ROOT / "src" / "urop" / "__init__.py",
    "vendored AutoTraceGT (agents/open_coding.py)": REPO_ROOT / "external" / "autotracegt_upstream" / "autotracegt" / "agents" / "open_coding.py",
    "vendored AutoTraceGT prompts": REPO_ROOT / "external" / "autotracegt_upstream" / "autotracegt" / "prompts" / "open_coding_system.md",
    "vendored tau-bench (for real tool schemas)": REPO_ROOT / "external" / "tau-bench" / "tau_bench" / "envs" / "retail" / "tools" / "__init__.py",
    "real smoke sample (3 trajectories)": REPO_ROOT / "data" / "tau_bench" / "tau_bench_train_3.jsonl",
    "smoke sample manifest": REPO_ROOT / "data" / "tau_bench" / "smoke_sample_manifest.json",
}
missing = {name: str(p) for name, p in REQUIRED_PATHS.items() if not p.exists()}
if missing:
    raise RuntimeError(
        f"Required paths missing from the attached bundle, refusing to proceed: {missing}. "
        "Rebuild the bundle per docs/kaggle_quickstart.md."
    )
print("All required paths present:")
for name, p in REQUIRED_PATHS.items():
    print(f"  OK  {name}: {p}")


## 3. Serving-stack install: explicit procedure, real evidence, preflight

**Real finding, verified against PyPI metadata:** every vLLM release
inspected (0.10.0-0.30.0) pins an EXACT torch version -- there is no
version that avoids a torch reinstall unless Kaggle's preinstalled torch
happens to exactly match one pin. Resolve this by measurement (below), not
by guessing.

**Version comparison uses `urop.env_check.torch_versions_compatible`**
(PEP 440 public-release comparison, ignoring a local `+cuXXX` build
suffix) -- a compatible match is still only ONE necessary check, not
sufficient proof vLLM will run.

**The real 2026-09-29 install produced dependency-resolver conflict
warnings against some of Kaggle's preinstalled packages (Google client
libraries, Gradio)** -- this environment is NOT conflict-free, and the
install cell below saves pip's own full stdout/stderr so those warnings
are recorded as evidence, not hidden. Nothing here reinstalls or upgrades
anything beyond the explicit `vllm==<version>` and
`requirements/kaggle-candidate.txt` -- those warnings are read and kept,
not resolved by installing more things.

Also in this section: a server entry-point check (does
`vllm.entrypoints.openai.api_server` even import and parse `--help`,
without launching a model) and the CUDA-linker workaround the real run
needed (`ld: cannot find -lcuda` during FlashInfer's kernel JIT compile) --
both previously ad hoc cells, now permanent and verified.


In [ ]:
# Explicit install procedure -- nothing here runs automatically.

INSTALL_EXTRA_DEPS = False  # flip only after reading requirements/kaggle-candidate.txt AND the cell output below

try:
    import torch
    installed_torch = torch.__version__
    print("torch already present:", installed_torch, "cuda:", torch.version.cuda,
          "available:", torch.cuda.is_available())
except ImportError:
    installed_torch = None
    print("torch not importable yet -- unusual for a Kaggle GPU image, double-check the environment.")

# tiktoken: small, standalone, needed only for the informational-only pre-check
# below -- installed explicitly and separately from the big candidate-deps
# install, so it does not depend on INSTALL_EXTRA_DEPS being flipped.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tiktoken>=0.7"], check=True)
import tiktoken
print("tiktoken installed:", tiktoken.__file__)


In [ ]:
# Look up which vLLM releases (if any) were built against the installed torch
# version -- a real network lookup against PyPI, not a hardcoded guess. Checks
# a bounded, recent shortlist (not all ~100 releases) to stay fast.
CANDIDATE_VLLM_VERSIONS_TO_CHECK = ["0.30.0", "0.27.1", "0.25.0", "0.22.1", "0.20.0", "0.18.1", "0.15.0", "0.10.0"]

def vllm_torch_pin(version: str) -> str | None:
    import urllib.request, json as _json
    with urllib.request.urlopen(f"https://pypi.org/pypi/vllm/{version}/json", timeout=10) as resp:
        info = _json.load(resp)["info"]
    for dep in info.get("requires_dist") or []:
        if dep.lower().startswith("torch==") and "vision" not in dep.lower() and "audio" not in dep.lower():
            return dep.split("==")[1].split(";")[0].strip()
    return None

pins = {}
for v in CANDIDATE_VLLM_VERSIONS_TO_CHECK:
    try:
        pins[v] = vllm_torch_pin(v)
    except Exception as exc:  # noqa: BLE001 -- report, keep checking the rest
        pins[v] = f"lookup failed: {exc}"

from urop.env_check import torch_versions_compatible

print(f"Installed torch: {installed_torch}")
print("vLLM version -> required torch (compared on the PEP 440 PUBLIC release only, "
      "ignoring e.g. a '+cu121' local build suffix -- see urop.env_check):")
for v, pin in pins.items():
    match = " <-- COMPATIBLE with installed torch" if torch_versions_compatible(installed_torch, pin) else ""
    print(f"  {v}: {pin}{match}")

matches = [v for v, pin in pins.items() if torch_versions_compatible(installed_torch, pin)]
if matches:
    print(f"\nCandidate (torch-compatible) vLLM version(s): {matches} -- reinstalling vLLM should NOT force a torch change.")
else:
    print("\nNo checked vLLM version is compatible with the installed torch. Installing ANY of them will "
          "reinstall torch -- a real risk to the existing CUDA build. Decide deliberately before proceeding.")

print("\nNOTE: a compatible torch pin is ONE necessary check, not sufficient proof vLLM will actually run -- "
      "GPU compatibility stays UNTESTED until Section 4's server actually starts.")


In [ ]:
CONFIRMED_VLLM_VERSION = None  # <- set explicitly (e.g. "0.18.1", the value verified 2026-09-29/30 -- see requirements/kaggle-verified-pins.txt) after reading the lookup above -- do not guess
INSTALL_EXTRA_DEPS = False  # actually execute the install below -- only after reviewing the printed plan

# If a previously-verified pins file exists, use it as a pip constraints
# file (`-c`) so the resolver can't silently pick different transitive
# versions than what was actually tested (2026-09-29/30 Kaggle run) --
# falls back to no constraints if this repo has none yet (first-ever run).
_VERIFIED_PINS = REPO_ROOT / "requirements" / "kaggle-verified-pins.txt"
_CONSTRAINTS_ARGS = ["-c", str(_VERIFIED_PINS)] if _VERIFIED_PINS.exists() else []

def _show_install_plan(vllm_version: str) -> dict | None:
    """Print exactly what pip would change -- including any torch/CUDA
    version change -- BEFORE anything is actually installed. Never silently
    replace torch/CUDA: this is the only place allowed to run the real
    `pip install` below, and only after this plan has been shown, and it
    REFUSES (returns None) if the plan would touch torch at all.
    """
    cmd = [
        sys.executable, "-m", "pip", "install", "--only-binary=:all:", "--dry-run", "--report", "-",
        *_CONSTRAINTS_ARGS,
        f"vllm=={vllm_version}", "-r", str(REPO_ROOT / "requirements" / "kaggle-candidate.txt"),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    (INSTALL_EVIDENCE_DIR / "install_preview_log.txt").write_text(result.stdout + "\n" + result.stderr, encoding="utf-8")
    if result.returncode != 0:
        print("Dry-run plan unavailable (pip may be too old for --dry-run/--report -- upgrade pip, or "
              "review requirements/kaggle-candidate.txt by hand before proceeding). pip stderr:")
        print(result.stderr[-2000:])
        return None
    report = json.loads(result.stdout)
    (INSTALL_EVIDENCE_DIR / "install_preview.json").write_text(result.stdout, encoding="utf-8")
    print(f"Proposed install plan for vllm=={vllm_version} (NOTHING installed yet; saved to install_preview.json):")
    for item in report.get("install", []):
        meta = item.get("metadata", {})
        name, new_version = meta.get("name"), meta.get("version")
        flag = " <-- TORCH VERSION CHANGE" if name and name.lower() == "torch" and new_version != installed_torch else ""
        print(f"  {name}: -> {new_version}{flag}")
    if any(i["metadata"]["name"].lower() == "torch" for i in report.get("install", [])):
        print("\nREFUSING: the plan above would replace/reinstall torch -- a real risk to the existing "
              "CUDA build. Adjust requirements/kaggle-candidate.txt or the constraints file so it doesn't, "
              "then re-run this cell before setting INSTALL_EXTRA_DEPS=True.")
        return None
    return report

def _run_real_install(vllm_version: str) -> None:
    """Actually installs -- vllm==<version> plus EXACTLY
    requirements/kaggle-candidate.txt, nothing broader, no --upgrade,
    constrained to previously-verified transitive pins when available.
    Saves the full evidence: the real pip --report JSON, the COMPLETE
    stdout+stderr (this is where pip's own dependency-resolver conflict
    warnings against Kaggle's preinstalled packages actually appear --
    e.g. Google client libraries and Gradio produced them in the real
    2026-09-29 run -- recorded here, not hidden), and a flat pins.txt.
    """
    report_path = INSTALL_EVIDENCE_DIR / "install_actual.json"
    log_path = INSTALL_EVIDENCE_DIR / "install_log.txt"
    cmd = [
        sys.executable, "-m", "pip", "install", "--only-binary=:all:",
        *_CONSTRAINTS_ARGS,
        f"vllm=={vllm_version}", "-r", str(REPO_ROOT / "requirements" / "kaggle-candidate.txt"),
        "--report", str(report_path),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    log_path.write_text(f"$ {' '.join(cmd)}\n\n--- stdout ---\n{result.stdout}\n\n--- stderr ---\n{result.stderr}\n", encoding="utf-8")
    print(f"Full install log saved to {log_path} -- READ IT. It is expected to contain pip's own dependency-"
          "resolver conflict warnings against some preinstalled packages; this environment is not conflict-free.")
    if result.returncode != 0:
        raise RuntimeError(f"pip install failed (exit {result.returncode}) -- see {log_path}")
    if report_path.exists():
        report = json.loads(report_path.read_text(encoding="utf-8"))
        assert not any(i["metadata"]["name"].lower() == "torch" for i in report.get("install", [])), (
            "torch was replaced despite the pre-install check -- stop, this should not have happened."
        )
        pins_text = "\n".join(f"{i['metadata']['name']}=={i['metadata']['version']}" for i in report.get("install", []))
        pins_path = INSTALL_EVIDENCE_DIR / "install_pins.txt"
        pins_path.write_text(pins_text + "\n", encoding="utf-8")
        resolved_path = INSTALL_EVIDENCE_DIR / "resolved_constraints.txt"
        resolved_path.write_text(
            (_VERIFIED_PINS.read_text(encoding="utf-8") + "\n" if _VERIFIED_PINS.exists() else "") + pins_text + "\n",
            encoding="utf-8",
        )
        print(f"Pins saved to {pins_path} ({len(report.get('install', []))} packages); resolved constraints saved to {resolved_path}.")

_install_plan = _show_install_plan(CONFIRMED_VLLM_VERSION) if CONFIRMED_VLLM_VERSION else None
if not CONFIRMED_VLLM_VERSION:
    print("Set CONFIRMED_VLLM_VERSION above first to see the install plan.")

if INSTALL_EXTRA_DEPS:
    if not CONFIRMED_VLLM_VERSION:
        raise RuntimeError("Set CONFIRMED_VLLM_VERSION explicitly first -- refusing to let pip pick a version for you.")
    if _install_plan is None:
        raise RuntimeError("No install plan was shown (dry-run failed, refused a torch replacement, or unavailable) -- refusing to install blind.")
    _run_real_install(CONFIRMED_VLLM_VERSION)
    import importlib
    vllm_mod = importlib.import_module("vllm")
    print(f"vllm {vllm_mod.__version__} importable. GPU compatibility is STILL UNTESTED until Section 4's "
          "server actually starts.")
else:
    print("INSTALL_EXTRA_DEPS=False -- nothing installed. Flip it only after reviewing the plan above.")


In [ ]:
# Server entry-point check + model-revision resolution.
# Verified once for real on Kaggle (2026-09-29) -- was an ad hoc cell then,
# now permanent. Model name is a standalone literal here (not read from the
# config cell below) so this diagnostic can run BEFORE that cell exists --
# keep the two in sync if you change the candidate model.
_REVISION_LOOKUP_MODEL = "Qwen/Qwen2.5-3B-Instruct"

result = subprocess.run(
    [sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--help"],
    capture_output=True, text=True, timeout=120,
)
(INSTALL_EVIDENCE_DIR / "server_import_check.txt").write_text(result.stdout + "\n" + result.stderr, encoding="utf-8")
print("Server entry-point exit code:", result.returncode)
if result.returncode != 0:
    print(result.stdout[-4000:])
    print(result.stderr[-6000:])
    raise RuntimeError("Server entry-point check failed; see install_evidence/server_import_check.txt.")
print("Server entry-point check passed.")

# Retrieve metadata only; no tokenizer or model weights.
with urllib.request.urlopen(f"https://huggingface.co/api/models/{_REVISION_LOOKUP_MODEL}", timeout=30) as response:
    model_info = json.load(response)
resolved_revision = model_info.get("sha")
assert resolved_revision, "Hugging Face returned no model revision."

print("\nModel:", _REVISION_LOOKUP_MODEL)
print("Resolved (candidate) revision:", repr(resolved_revision))
print("This PROPOSES a revision -- copy it into CANDIDATE_MODEL_REVISION in the config cell "
      "below yourself; it is never set automatically.")

(INSTALL_EVIDENCE_DIR / "model_selection.json").write_text(
    json.dumps({"model": _REVISION_LOOKUP_MODEL, "resolved_revision": resolved_revision}, indent=2), encoding="utf-8",
)


In [ ]:
# CUDA-linker workaround -- REQUIRED before starting the server on this
# Kaggle image. The real 2026-09-29 run's first launch failed while
# FlashInfer JIT-compiled a kernel: "/usr/bin/ld: cannot find -lcuda".
# `ldconfig` alone did not locate the driver library; inspecting
# /proc/self/maps after loading libcuda.so.1 did. This symlinks the ACTUAL
# loaded driver library into a writable directory and points LIBRARY_PATH
# at it -- installs nothing, downloads nothing -- then verifies the fix
# with a real link test before trusting it.
import ctypes

if not sys.platform.startswith("linux"):
    print(f"Platform is {sys.platform!r}, not Linux -- this workaround is Linux/Kaggle-specific "
          "(libcuda.so, /proc/self/maps, a POSIX c++ link test) and does not apply here. Skipping.")
elif shutil.which("nvidia-smi") is None:
    print("No nvidia-smi found -- skipping the CUDA-linker workaround (nothing to link against). "
          "Run this cell again once a GPU is actually attached.")
else:
    cuda_driver = ctypes.CDLL("libcuda.so.1")  # load the existing driver; installs nothing

    candidates = []
    for line in Path("/proc/self/maps").read_text().splitlines():
        fields = line.split()
        if fields and fields[-1].startswith("/"):
            p = Path(fields[-1])
            if p.name.startswith("libcuda.so") and p.is_file():
                candidates.append(p.resolve())
    for folder in ["/usr/lib/x86_64-linux-gnu", "/usr/lib64", "/usr/local/nvidia/lib64", "/usr/local/nvidia/lib"]:
        for p in Path(folder).glob("libcuda.so*"):
            if p.is_file() and "stubs" not in str(p.resolve()):
                candidates.append(p.resolve())

    assert candidates, "Driver path still not found. Record this output before doing anything else."
    driver_library = candidates[0]
    print("CUDA driver library:", driver_library)

    link_dir = Path("/kaggle/working/urop-cuda-link")
    link_dir.mkdir(exist_ok=True)
    link = link_dir / "libcuda.so"
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        raise RuntimeError(f"Unexpected existing file: {link}")
    link.symlink_to(driver_library)

    existing = os.environ.get("LIBRARY_PATH", "")
    os.environ["LIBRARY_PATH"] = str(link_dir) + (os.pathsep + existing if existing else "")

    source = link_dir / "link_check.cpp"
    source.write_text('extern "C" int cuInit(unsigned int);\nint main() { return cuInit(0); }\n')
    compiler = shutil.which("c++")
    assert compiler, "C++ compiler not found."
    result = subprocess.run(
        [compiler, str(source), "-lcuda", "-o", str(link_dir / "link_check")],
        capture_output=True, text=True, timeout=60,
    )
    print(result.stdout)
    print(result.stderr)
    assert result.returncode == 0, "Link test failed; record this output before doing anything else."
    print("CUDA linker check PASSED -- LIBRARY_PATH is set for the rest of this kernel session, "
          "including the server subprocess Section 4 launches (it inherits this process's environment).")


## 4-5. Config, then ONE protected lifecycle: serve + tokenizer + agent + code

Server startup, the independent `DeadlineSupervisor`, tokenizer loading,
agent construction, and the whole coding loop share ONE
`try/except/finally` -- ANY failure at ANY stage tears down the owned
server process GROUP and saves a stop/error manifest.

**New this revision:**
- `OUTPUT_MODE` selects between upstream's own freeform `===CODES===`
  format (`"upstream_codes_block"`) and the opt-in
  `urop.structured_coding` JSON-schema-constrained mode
  (`"structured_json"`) -- see that module's docstring for why the
  structured mode exists and what it changes (and does not change) about
  the actual coding instructions. Recorded in `GENERATION_SETTINGS` and
  folded into the cache key automatically (settings_hash changes with it).
- `urop.evidence.EvidenceLog`/`CodingContext` are wired into
  `install_budget_guard` -- every real request/response (or block, or
  error) is saved to `runs/kaggle_smoke/<run_id>/evidence/` the moment it
  happens, independent of whether the trajectory it belongs to ever
  finishes.
- A trajectory-level `PromptTooLongError`/`StructuredOutputError` no
  longer discards chunks already completed earlier in the SAME trajectory
  (`exc.partial_result`) -- those are saved and classified too, not lost.
- Requires an explicit, real model revision (`CANDIDATE_MODEL_REVISION`) --
  refuses to start with a floating ref.


In [ ]:
from urop.budgets import Budget, BudgetExhausted
from urop.process_control import (
    DeadlineSupervisor, ServerExitedDuringStartup, ServerNotReady,
    shutdown_process_group, start_owned_process, wait_for_server_ready,
)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # single T4 -- leave GPU 1 unused (still bills for both -- see intro)

CANDIDATE_MODEL = "Qwen/Qwen2.5-3B-Instruct"  # must match _REVISION_LOOKUP_MODEL above if you change it

# One immutable, resolved model revision -- REQUIRED before RUN_SERVER=True.
# Copy the value the entry-point/revision cell above printed and saved to
# install_evidence/model_selection.json -- never resolved automatically here.
# Verified 2026-09-29/30: "aa8e72537993ba99e69dfaafa59ed015b17504d1" (reference
# only -- re-resolve and confirm it's still current, don't assume it).
CANDIDATE_MODEL_REVISION: str | None = None

# Output mode -- see Section 4-5's markdown above.
#   "upstream_codes_block": AutoTraceGT's own freeform format, unmodified.
#   "structured_json":      opt-in JSON-schema-constrained mode (urop.structured_coding),
#                            a methodological adaptation -- see docs/methods_and_deviations.md.
OUTPUT_MODE = "structured_json"

GENERATION_SEED = 0  # explicit, recorded, sent as extra_body={"seed": ...} on every request below

LOCAL_SERVER_PORT = 8000
SERVER_READY_TIMEOUT_S = 600

# Enforced everywhere below, per the compute plan (docs/compute_budget.md):
CODING_CONCURRENCY = 1
CANDIDATE_CONTEXT_LIMIT = 8192
CANDIDATE_MAX_NEW_TOKENS = 1024
CODING_TEMPERATURE = 1  # matches AutoTraceGT paper's own reported default (docs/source_audit.md)
JSON_RETRIES_TOTAL_ATTEMPTS = 2  # upstream MAX_JSON_RETRIES counts TOTAL attempts: 1 initial + 1 repair = 2 -- freeform mode only, structured mode has no repair round (see urop.structured_coding)
os.environ["JSON_RETRIES"] = str(JSON_RETRIES_TOTAL_ATTEMPTS)  # must be set before the FIRST upstream import this session

# A whole-TRAJECTORY parse/schema/empty/quote failure gets exactly this many
# TOTAL attempts before being permanently marked invalid and reported --
# never rerun indefinitely on a later resume. Set to 1 (2026-09-30): this
# bounded one-trajectory diagnostic allows exactly one attempt, no automatic
# retry -- raise deliberately, with justification, for a larger run.
# See ResumeCache.is_exhausted below.
MAX_RESULT_LEVEL_ATTEMPTS = 1

# THIS diagnostic run's budget starts being CONSTRUCTED immediately before
# Section 5's protected lifecycle actually runs (next cell after data
# loading) -- not here -- so time spent reading/editing this config cell
# does not silently consume it. See that cell for SMOKE_BUDGET's real
# construction.
SMOKE_BUDGET_MAX_SECONDS = 30 * 60      # HTTP-call-level: bounds one request at a time
SMOKE_BUDGET_MAX_CALLS = 1              # this diagnostic run is scoped to exactly one real request (2026-09-30) -- raise deliberately, with justification, for a larger run
SMOKE_WALLCLOCK_DEADLINE_S = 35 * 60    # independent supervisor deadline, slightly looser than the HTTP budget

RUN_SERVER = False  # flip only once ready to actually spend GPU time
server_process = None
supervisor = None


### Select data, then run the real open-coding pass

**Requires real data -- no synthetic fallback.** If
`data/tau_bench/tau_bench_train_3.jsonl` is missing, Section 2's path
validation already stopped the notebook before this point.

**This run is scoped to exactly ONE trajectory** -- `instance_id="13"`,
`run_id="trial_0"` -- the one confirmed malformed under freeform mode on
2026-09-29 (unescaped quotes inside a tool-call quotation broke
`json.loads`). The next cell asserts exactly one record is selected;
change `SELECTED_INSTANCE_ID`/`SELECTED_RUN_ID` deliberately, not by
deleting the assertion.


In [ ]:
from urop.data import load_and_clean_jsonl

real_data_path = REPO_ROOT / "data" / "tau_bench" / "tau_bench_train_3.jsonl"
if not real_data_path.exists():
    raise RuntimeError(f"{real_data_path} missing -- this notebook requires real converted data, no synthetic fallback.")

all_records, load_report = load_and_clean_jsonl(real_data_path, domain="tau_bench_retail")
print(f"Loaded {load_report.accepted} real records from {real_data_path.name}.")

# One-trajectory diagnostic selection -- see this section's markdown above.
SELECTED_INSTANCE_ID = "13"
SELECTED_RUN_ID = "trial_0"
records = [r for r in all_records if r.instance_id == SELECTED_INSTANCE_ID and r.run_id == SELECTED_RUN_ID]
assert len(records) == 1, (
    f"expected exactly 1 record for instance_id={SELECTED_INSTANCE_ID!r} run_id={SELECTED_RUN_ID!r}, "
    f"got {len(records)} -- check data/tau_bench/tau_bench_train_3.jsonl and the IDs above."
)
print(f"Selected exactly 1 record for this diagnostic run: {records[0].composite_id}")


In [ ]:
from urop.autotrace_adapter import (
    classify_trajectory_coding, code_trajectory_preserving_partial_results,
    ensure_upstream_on_path, hash_prompt_templates, install_budget_guard,
    register_local_endpoint, validate_coding_quotes, PromptTooLongError,
)
from urop.budgets import ResumeCache, cache_key
from urop.evidence import CodingContext, EvidenceLog
from urop.manifests import build_run_manifest, save_manifest, sha256_json
from urop.reporting import build_outcome_report
from urop.structured_coding import StructuredOutputError, code_record_structured
import importlib.metadata


def _installed_version(pkg: str) -> str | None:
    try:
        return importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        return None


if OUTPUT_MODE not in ("upstream_codes_block", "structured_json"):
    raise ValueError(f"OUTPUT_MODE={OUTPUT_MODE!r} not recognized -- see the config cell above.")

run_id = time.strftime("kaggle_smoke_%Y%m%dT%H%M%SZ", time.gmtime())
out_dir = REPO_ROOT / "runs" / "kaggle_smoke" / run_id  # under /kaggle/working -- writable, git-ignored
out_dir.mkdir(parents=True, exist_ok=True)
evidence_dir = out_dir / "evidence"  # one JSON per real/blocked request -- see urop.evidence

coded_results_by_id: dict[str, dict] = {}   # composite_id -> TrajectoryCoding.to_dict(), fresh OR loaded from cache
skipped_via_cache: list[str] = []
context_limit_blocks: list[dict] = []       # requests the token guard refused to send -- NOT real attempts
schema_failures: list[str] = []             # StructuredOutputError composite_ids -- distinct from context-limit blocks
invalid_results: list[str] = []             # parse-failed / empty / quote-invalid, bounded retry, then permanently reported
quote_validation_reports: dict[str, object] = {}
stop_reason: str | None = None
_setup_error: BaseException | None = None
server_startup_success = False
GENERATION_SETTINGS = None                  # filled in below once resolvable
SMOKE_BUDGET = None                         # constructed immediately before use, below -- not in the config cell


def _save_stop_manifest() -> None:
    """Always called from `finally` below -- runs on a clean completion, a
    budget cutoff, AND a failure during setup itself, before a single
    coding call ever happened. Builds a `RunOutcomeReport` so "the loop
    finished" can never be read as "the loop produced usable output" --
    see urop.reporting.
    """
    gpu_info = {}
    try:
        import torch
        if torch.cuda.is_available():
            gpu_info = {"gpu_name": torch.cuda.get_device_name(0), "gpu_free_mem_gb": torch.cuda.mem_get_info(0)[0] / 1e9}
    except ImportError:
        pass

    length_limited_count = 0
    if evidence_dir.exists():
        for p in evidence_dir.glob("evidence_*.json"):
            try:
                if json.loads(p.read_text(encoding="utf-8")).get("finish_reason") == "length":
                    length_limited_count += 1
            except (json.JSONDecodeError, OSError):
                pass

    final_stop_reason = stop_reason or (
        f"error_during_setup_or_coding: {_setup_error!r}" if _setup_error is not None else "completed"
    )
    outcome = build_outcome_report(
        run_id=run_id,
        server_startup_success=server_startup_success,
        stop_reason=final_stop_reason,
        records_loaded=len(records),
        real_attempts_made=SMOKE_BUDGET.calls_made if SMOKE_BUDGET is not None else 0,
        context_limit_blocks=len(context_limit_blocks),
        coded_results_by_composite_id=coded_results_by_id,
        quote_validation_reports=quote_validation_reports,
        length_limited_response_count=length_limited_count,
        server_attempted=RUN_SERVER,
        failed_record_ids=set(invalid_results + schema_failures),
    )

    config = {
        "candidate_model": CANDIDATE_MODEL,
        "candidate_model_revision": CANDIDATE_MODEL_REVISION,
        "output_mode": OUTPUT_MODE,
        "run_server": RUN_SERVER,
        "generation_settings": GENERATION_SETTINGS,
        "records_skipped_via_cache": skipped_via_cache,
        "invalid_results": invalid_results,
        "schema_failures": schema_failures,
        "cuda_linker_library_path": os.environ.get("LIBRARY_PATH", ""),
        "context_limit_block_details": context_limit_blocks,
        "wallclock_supervisor_deadline_seconds": SMOKE_WALLCLOCK_DEADLINE_S,
        "wallclock_supervisor_fired": supervisor.fired if supervisor is not None else False,
        "outcome_report": outcome.to_dict(),
        **gpu_info,
    }
    manifest = build_run_manifest(
        run_id=run_id, stage="open_coding_smoke", config=config,
        input_files=[real_data_path], backend_label="local" if RUN_SERVER else "n/a",
    )
    manifest.stop_reason = final_stop_reason
    save_manifest(manifest, out_dir / "manifest.json")
    print("\n=== Outcome report (automated evidence checks -- NOT human-validated quality) ===")
    print(json.dumps(outcome.to_dict(), indent=2))
    print(f"\nFull manifest (incl. generation settings, evidence dir: {evidence_dir}) saved to "
          f"{out_dir / 'manifest.json'} -- git-ignored, copy it out before the session ends (Section 9).")


def _preserve_partial(exc, r, attempt_n, key, reason_prefix) -> bool:
    """Shared handling for a mid-trajectory block/schema-failure: preserve
    whatever EARLIER chunks of this SAME trajectory already completed
    (`exc.partial_result`, set by code_trajectory_preserving_partial_results
    / code_record_structured) instead of discarding them -- the real
    "trajectory 102" bug this project fixed (2026-09-30, see
    docs/experiment_log.md). Returns True iff a partial result was found
    and saved (caller should skip its own no-partial handling in that case).
    """
    partial = getattr(exc, "partial_result", None)
    if partial is None or not partial.chunk_codings:
        return False
    partial_dict = partial.to_dict()
    result_path = out_dir / f"open_coding_{r.instance_id}_{r.run_id}_attempt{attempt_n}_partial.json"
    result_path.write_text(json.dumps(partial_dict, indent=2), encoding="utf-8")
    resume_cache.mark_invalid(key, reason=f"{reason_prefix}: {exc}", result_path=str(result_path))
    print(f"  PARTIAL ({reason_prefix}, {len(partial.chunk_codings)} earlier chunk(s) "
          f"preserved) {r.composite_id} -> {result_path.name}")
    return True


try:
    if RUN_SERVER:
        if not CANDIDATE_MODEL_REVISION:
            raise RuntimeError(
                "CANDIDATE_MODEL_REVISION must be set explicitly (a real commit hash) before "
                "RUN_SERVER=True -- refusing to let the server and the tokenizer each silently "
                "resolve a floating ref, possibly to DIFFERENT weights on either side."
            )

        server_process = start_owned_process([
            sys.executable, "-m", "vllm.entrypoints.openai.api_server",
            "--model", CANDIDATE_MODEL,
            "--revision", CANDIDATE_MODEL_REVISION,
            "--tokenizer-revision", CANDIDATE_MODEL_REVISION,
            "--dtype", "float16",
            "--enable-auto-tool-choice",
            "--tool-call-parser", "hermes",
            "--port", str(LOCAL_SERVER_PORT),
            "--max-model-len", str(CANDIDATE_CONTEXT_LIMIT),
            "--gpu-memory-utilization", "0.85",
            "--seed", str(GENERATION_SEED),
        ])
        supervisor = DeadlineSupervisor(server_process, SMOKE_WALLCLOCK_DEADLINE_S)
        supervisor.start()
        print(f"Independent wall-clock supervisor started: will force-stop the whole server process "
              f"group after {SMOKE_WALLCLOCK_DEADLINE_S}s no matter what the main thread is doing.")

        elapsed = wait_for_server_ready(
            server_process,
            lambda: check_reachable(f"http://localhost:{LOCAL_SERVER_PORT}/health") == "reachable",
            timeout_seconds=SERVER_READY_TIMEOUT_S,
        )
        server_startup_success = True
        print(f"Server ready after {elapsed:.1f}s.")

        register_local_endpoint(base_url=f"http://localhost:{LOCAL_SERVER_PORT}/v1")
        ensure_upstream_on_path()
        from agents.open_coding import OpenCodingAgent  # real upstream agent, not a stub
        from transformers import AutoTokenizer

        qwen_tokenizer = AutoTokenizer.from_pretrained(CANDIDATE_MODEL, revision=CANDIDATE_MODEL_REVISION)
        installed_versions = {pkg: _installed_version(pkg) for pkg in ("torch", "transformers", "vllm", "openai")}

        agent = OpenCodingAgent(
            model=CANDIDATE_MODEL, endpoint="local",
            chunk_size=50, concurrency=CODING_CONCURRENCY,
            max_tokens=CANDIDATE_MAX_NEW_TOKENS, temperature=CODING_TEMPERATURE,
            extra_body={"seed": GENERATION_SEED},  # OpenCodingAgent has no seed param of its own -- injected via extra_body
            offline_dir=None,
        )

        # THE request budget -- constructed here, immediately before use, so
        # time spent reading/editing the config cell above never counts
        # against it (a real fix: previously `Budget(...)` was constructed
        # IN the config cell, so any delay before this cell ran was silently
        # eaten out of the 30-minute window).
        SMOKE_BUDGET = Budget(max_seconds=SMOKE_BUDGET_MAX_SECONDS, max_calls=SMOKE_BUDGET_MAX_CALLS)

        GENERATION_SETTINGS = {
            "model": CANDIDATE_MODEL,
            "model_revision": CANDIDATE_MODEL_REVISION,
            "output_mode": OUTPUT_MODE,
            "generation_seed": GENERATION_SEED,
            "temperature": CODING_TEMPERATURE,
            "max_tokens": CANDIDATE_MAX_NEW_TOKENS,
            "chunk_size": agent.chunk_size,
            "concurrency": CODING_CONCURRENCY,
            "context_limit": CANDIDATE_CONTEXT_LIMIT,
            "json_retries_total_attempts": JSON_RETRIES_TOTAL_ATTEMPTS if OUTPUT_MODE == "upstream_codes_block" else 1,
            "max_result_level_attempts": MAX_RESULT_LEVEL_ATTEMPTS,
            "endpoint": "local",
            "installed_versions": installed_versions,
        }
        if OUTPUT_MODE == "structured_json":
            from urop.structured_coding import CODING_RESULT_SCHEMA
            GENERATION_SETTINGS["structured_output_schema"] = CODING_RESULT_SCHEMA  # part of settings_hash -> cache key
        settings_hash = sha256_json(GENERATION_SETTINGS)
        print("Generation settings (explicit, recorded):", json.dumps(GENERATION_SETTINGS, indent=2))

        evidence_log = EvidenceLog(evidence_dir)
        coding_context = CodingContext()
        install_budget_guard(
            agent, SMOKE_BUDGET,
            tokenizer=qwen_tokenizer, context_limit=CANDIDATE_CONTEXT_LIMIT,
            reserved_output_tokens=CANDIDATE_MAX_NEW_TOKENS, max_sdk_retries=0,
            evidence_log=evidence_log, coding_context=coding_context,
            output_mode=OUTPUT_MODE, model_revision=CANDIDATE_MODEL_REVISION,
            generation_settings_hash=settings_hash,
        )
        print(f"OpenCodingAgent ready: model={agent.model} revision={CANDIDATE_MODEL_REVISION} "
              f"output_mode={OUTPUT_MODE} concurrency={agent.concurrency} chunk_size={agent.chunk_size} "
              f"temperature={CODING_TEMPERATURE} seed={GENERATION_SEED}")

        prompt_hash = hash_prompt_templates("open_coding_system", "open_coding_user")
        if OUTPUT_MODE == "structured_json":
            # Fold the ACTUAL structured system prompt (including the
            # diagnostic-clarification text appended in urop.structured_coding)
            # into the hash too -- hash_prompt_templates only covers upstream's
            # own freeform templates, so a change to the structured-mode prompt
            # text alone would not otherwise invalidate stale cache entries.
            from urop.structured_coding import build_structured_system_prompt
            prompt_hash = sha256_json({"upstream": prompt_hash, "actual_system": build_structured_system_prompt()})
        resume_cache = ResumeCache(out_dir.parent / "resume_index.json")  # shared across runs, one dir up from this run_id

        code_fn = code_record_structured if OUTPUT_MODE == "structured_json" else code_trajectory_preserving_partial_results

        for r in records:
            key = cache_key(
                model=CANDIDATE_MODEL, model_revision=CANDIDATE_MODEL_REVISION,
                prompt_hash=prompt_hash, settings_hash=settings_hash,
                data_hash=r.source_record_hash, stage="open_coding",
            )

            if resume_cache.is_done(key):
                cached_path = resume_cache.result_path_for(key)
                cached_dict = json.loads(cached_path.read_text(encoding="utf-8"))
                # Revalidate a cache hit against the CURRENT (possibly stricter)
                # rules before trusting it -- an entry marked done under an
                # earlier, looser validator must not be silently reused now.
                validation = validate_coding_quotes(cached_dict, trajectory=r.trajectory, chunk_size=agent.chunk_size)
                if classify_trajectory_coding(cached_dict) != "valid" or validation.total_codes == 0 or validation.invalid_codes:
                    resume_cache.mark_invalid(key, reason="cache failed revalidation", result_path=str(cached_path))
                    raise RuntimeError(
                        f"Cached result for {r.composite_id} failed revalidation under current rules; "
                        "preserved on disk, not reused. Stop and inspect before proceeding."
                    )
                coded_results_by_id[r.composite_id] = cached_dict  # LOADED, not just skipped -- counts in this run's report
                quote_validation_reports[r.composite_id] = validation
                skipped_via_cache.append(r.composite_id)
                print(f"  SKIP (cache hit, revalidated, loaded {cached_path.name}) {r.composite_id}")
                continue

            if resume_cache.is_exhausted(key, max_attempts=MAX_RESULT_LEVEL_ATTEMPTS):
                invalid_results.append(r.composite_id)
                print(f"  SKIP (previously invalid, {MAX_RESULT_LEVEL_ATTEMPTS}/{MAX_RESULT_LEVEL_ATTEMPTS} "
                      f"attempts already used) {r.composite_id}")
                continue

            record_dict = {
                "instance_id": r.instance_id, "run_id": r.run_id,
                "context": r.context, "trajectory": r.trajectory, "status": r.status,
            }
            coding_context.begin_record(r.composite_id)
            attempt_n = resume_cache.record_attempt(key)
            try:
                coding = code_fn(agent, record_dict)
            except BudgetExhausted as exc:
                stop_reason = exc.reason
                print(f"Budget exhausted ({stop_reason}) -- stopping. {len(coded_results_by_id)} already saved, "
                      f"{SMOKE_BUDGET.calls_made} real attempts made in total.")
                break
            except StructuredOutputError as exc:
                # Genuine schema/parse failure -- distinct from a context-limit
                # block: NOT counted as a real attempt if it never reached
                # bounded_llm_call, tracked separately from PromptTooLongError.
                schema_failures.append(r.composite_id)
                if not _preserve_partial(exc, r, attempt_n, key, "schema failure mid-trajectory"):
                    resume_cache.mark_invalid(key, reason=str(exc), result_path=str(evidence_dir))
                    print(f"  SCHEMA FAILURE {r.composite_id}: {exc}; response evidence preserved in {evidence_dir}.")
                continue
            except PromptTooLongError as exc:
                # Blocked by the context-limit guard -- NOT counted as a real
                # attempt (it never reached bounded_llm_call).
                context_limit_blocks.append({"composite_id": r.composite_id, "error": str(exc)})
                if not _preserve_partial(exc, r, attempt_n, key, "blocked mid-trajectory"):
                    print(f"  SKIP (blocked before any chunk completed) {r.composite_id}: {exc}")
                continue

            coding_dict = coding.to_dict()
            classification = classify_trajectory_coding(coding_dict)
            result_path = out_dir / f"open_coding_{r.instance_id}_{r.run_id}_attempt{attempt_n}.json"
            result_path.write_text(json.dumps(coding_dict, indent=2), encoding="utf-8")  # raw output ALWAYS preserved, valid or not

            if classification == "parse_failed":
                resume_cache.mark_invalid(key, reason="unparseable/schema-invalid model output", result_path=str(result_path))
                invalid_results.append(r.composite_id)
                print(f"  INVALID (parse/schema failed, attempt {attempt_n}/{MAX_RESULT_LEVEL_ATTEMPTS}) "
                      f"{r.composite_id} -> {result_path.name}")
                continue

            coded_results_by_id[r.composite_id] = coding_dict
            validation = validate_coding_quotes(coding_dict, trajectory=r.trajectory, chunk_size=GENERATION_SETTINGS["chunk_size"])
            quote_validation_reports[r.composite_id] = validation
            # Empty output remains structurally valid but is NOT usable evidence
            # and must NOT be cached as done (2026-09-30 fix -- an earlier
            # version incorrectly counted a genuinely empty result as usable,
            # since an empty codes list trivially has zero invalid quotes too).
            if validation.total_codes == 0:
                resume_cache.mark_invalid(key, reason="empty codes; no usable evidence example", result_path=str(result_path))
                invalid_results.append(r.composite_id)
                print(f"  EMPTY OUTPUT (valid structure, zero behaviour labels, attempt {attempt_n}/{MAX_RESULT_LEVEL_ATTEMPTS}) "
                      f"{r.composite_id} -> {result_path.name}")
            elif validation.invalid_codes:
                resume_cache.mark_invalid(key, reason="quote or step validation failed", result_path=str(result_path))
                invalid_results.append(r.composite_id)
                print(f"  coded {r.composite_id}: {len(coding.chunk_codings)} chunks, "
                      f"{validation.valid_codes}/{validation.total_codes} codes with verbatim quotes "
                      f"({len(validation.invalid_codes)} invalid -- NOT cached as done) -> {result_path.name}")
            else:
                resume_cache.mark_done(key, result_path=str(result_path))
                print(f"  coded {r.composite_id}: {len(coding.chunk_codings)} chunks, "
                      f"{validation.valid_codes}/{validation.total_codes} codes with verbatim quotes -> {result_path.name}")
    else:
        print("RUN_SERVER=False -- nothing to register, nothing to run. This cell is a no-op by design.")
except Exception as exc:
    _setup_error = exc
    stop_reason = stop_reason or f"error_during_setup_or_coding: {exc!r}"
    raise
finally:
    # Runs on EVERY exit path: a clean loop completion, a budget cutoff
    # (`break` above), or an exception at ANY stage -- server launch,
    # supervisor start, wait_for_server_ready, tokenizer load, agent
    # construction, or the coding loop itself.
    if supervisor is not None:
        supervisor.cancel()
    if server_process is not None:
        result = shutdown_process_group(server_process, wait_seconds=30)
        print(f"Server process group shutdown (finally block): {result.how}")
    _save_stop_manifest()


## 6. Report

**Reporting now happens automatically inside Section 5's `finally` block**
(`_save_stop_manifest`, previous cell), built via
`urop.reporting.build_outcome_report` -- so "the loop finished" can never
be read as "the loop produced usable output". The cell below just re-reads
and re-prints the saved manifest for convenience.


In [ ]:
manifest_path = out_dir / "manifest.json"
print(f"Reading back {manifest_path} (already saved by Section 5's finally block):\n")
print(manifest_path.read_text(encoding="utf-8"))


## 7. Export results and evidence before the session ends

`runs/`, `install_evidence/`, and everything under `REPO_ROOT` is
git-ignored and lives only in this Kaggle session's storage -- not
permanent. This packages the coding results/manifests AND the install
evidence (pip report, log, pins, entry-point check, model selection, CUDA-
linker output) into one zip under `/kaggle/working` you can download from
the output pane. **This zip is private evidence -- do not commit it, do
not attach it to the public repo or either packaged bundle.**


In [ ]:
import zipfile
from IPython.display import FileLink, display

_export_root = Path("/kaggle/working")
_runs_dir = REPO_ROOT / "runs" / "kaggle_smoke"
_archive = _export_root / "urop_smoke_debug.zip"

with zipfile.ZipFile(_archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    if _runs_dir.is_dir():
        for path in _runs_dir.rglob("*"):
            if path.is_file():
                bundle.write(path, path.relative_to(REPO_ROOT))
    if INSTALL_EVIDENCE_DIR.is_dir():
        for path in INSTALL_EVIDENCE_DIR.rglob("*"):
            if path.is_file():
                bundle.write(path, Path("install_evidence") / path.relative_to(INSTALL_EVIDENCE_DIR))

print(f"Packaged {_archive} -- private evidence, do not commit or publish.")
display(FileLink(str(_archive)))


## 8. Optional next step (SEPARATELY GATED): single-batch axial-coding smoke

**HARD-DISABLED for this first run** (see the cell below) -- it currently
reuses Section 5's `server_process`/`supervisor` instead of having its OWN
protected lifecycle, so flipping `RUN_AXIAL_SMOKE` before that exists would
run unprotected. Implement a dedicated lifecycle for this section
(mirroring Section 5's) before ever setting it `True`.

**Renamed from `RUN_FULL_PIPELINE`** -- that name overstated its scope.
Were it enabled, this would run `AxialCodingAgent.code_batch()` **once**,
on the one small batch of open-coding results Section 5 just produced. It
is NOT the full AutoTraceGT loop -- `PostAxialEvalAgent`, `CodeManagerAgent`
(the actual saturation loop), and `SelectiveCodingAgent` are not run
anywhere in this notebook. A one-trajectory, single-round, stage-2-only
diagnostic cannot reach or imply saturation, a codebook, or a theory.


In [ ]:
RUN_AXIAL_SMOKE = False  # HARD-DISABLED for this first run -- see the raise below and this section's markdown

AXIAL_SMOKE_BUDGET = Budget(max_seconds=30 * 60, max_calls=10)  # kept defined for when this IS implemented

if RUN_AXIAL_SMOKE:
    raise RuntimeError(
        "RUN_AXIAL_SMOKE is disabled for this first run: axial coding currently reuses Section "
        "5's server_process/supervisor instead of having its OWN protected lifecycle (server "
        "startup + independent deadline supervisor + try/finally cleanup + stop manifest) -- "
        "flipping this flag before that exists risks an unprotected, unsupervised run. Implement "
        "a dedicated lifecycle for this section (mirroring Section 5's) before ever setting this "
        "True."
    )
else:
    print("RUN_AXIAL_SMOKE=False (hard-disabled for this first run) -- stages 2-5 beyond open coding not run.")


## 9. Shutdown — always run this, even if earlier cells failed

Section 5 already terminates the server process GROUP and cancels the
independent deadline supervisor in a `finally` block on any exception, so
this is a safety net (idempotent), not the only thing standing between you
and wasted GPU quota. **An ALLOCATED GPU session keeps consuming your
weekly quota for as long as it stays up, whether or not it is doing any
work at that moment -- stopping the server process here does NOT by
itself end the Kaggle session or stop the quota meter. You must also use
Kaggle's own Stop Session control below.** Selecting "GPU T4 x2" bills for
both GPUs for the whole session even though this notebook only ever uses
one.


In [ ]:
if supervisor is not None:
    supervisor.cancel()

if server_process is not None:
    result = shutdown_process_group(server_process, wait_seconds=30)
    print(f"Server shutdown: {result.how} (returncode={result.returncode})")
else:
    print("No server process was started (or it was already cleaned up by a finally block above).")

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("CUDA cache cleared.")
except ImportError:
    pass

print(
    "Stopping the server process above does NOT stop the Kaggle session or its quota meter. "
    "Now stop the SESSION itself: the menu in the top-right of the notebook editor -> "
    "'Stop Session' (or Save Version without keeping it running). Confirm the GPU indicator "
    "goes to 0 before closing the tab -- an idle allocated GPU still consumes your weekly quota."
)
